# Lab 2: GPT from scratch

In this lab, you will dive into the inner workings of the GPT architecture. You will walk through a complete implementation of the architecture in PyTorch, instantiate this implementation with pre-trained weights, and put the resulting model to the test by generating text. At the end of this lab, you will understand the building blocks of the GPT architecture and how they are connected.

*Tasks you can choose for the oral exam are marked with the graduation cap 🎓 emoji.*

In [1]:
from dataclasses import dataclass

import torch
import torch.nn as nn

## Part 1: GPT architecture

GPT-2 was first described by [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). To faithfully implement the model, one needs to also read the earlier paper by [Radford et al. (2018)](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf). Another important source of information is the code released by OpenAI, which is available on GitHub ([link](https://github.com/openai/gpt-2)).

The GPT architecture is made up of a stack of Transformer blocks. Each block has two main parts: one handles multi-head self-attention, and the other is a feed-forward network. Before these parts do their work, their input undergoes layer normalisation, and residual connections are added to help the model learn more effectively. The input to the architecture is a sequence of token IDs; these are turned into embeddings and augmented with information about the absolute position of each token in the sequence. The output layer converts the internal representations into logit scores for every token in the vocabulary.

### Model configuration

[Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) present four increasingly larger GPT models based on the same architecture. Here, we will implement the smallest of these, characterised by the following hyperparameters:

In [2]:
@dataclass
class Config:
    n_vocab: int = 50_257
    n_ctx: int = 1024
    n_embd: int = 768
    n_head: int = 12
    n_layer: int = 12

#### 🧩 Task 2.01: Model configuration

Explain the purpose of these hyperparameters. In particular, where does the number 50,257 come from?

⚙️ Hyperparameter Breakdown

1. n_vocab: This is number of unique tokens the model works function like a dictionary when translating.
    1. 256 = when tokenizing a copora with BPE with begin with base vocab of 256 tokens which is biscally all the ASCCI characters.
    2. 1 = This the special character with represent the end of a document.
    3. 50000 = This the result of running BPE on the copora, 50000 tokens.

2. n_context = this is the size of the context window which biscally defines the number tokens that the model proccess in ecah step.
3. e_embd = The size of the vector that represents each token, larger allows us to capture more complexity. 

4. n_head = the number of heads in out attention mechanism, each head focuses on a specific subtask like gramer, strcuture etc...
5. n_layers = the number of transformer blocks stacked on top of each other.


### GELU activation function

We start by implementing the feed-forward network. This is a standard two-layer network with a Gaussian Error Linear Unit (GELU) activation function ([Hendrycks and Gimpel, 2016](https://doi.org/10.48550/arXiv.1606.08415)).

The GELU is a smooth version of the rectified linear unit (ReLU) that weights inputs by their value under the cumulative distribution function of the standard Gaussian. This function is commonly denoted by $\Phi$. For example, $\text{GELU}(0{.}5) = 0{.}5 \cdot \Phi(0{.}5) \approx 0{.}5 \cdot 0{.}6915 = 0{.}3457$ because approximately 69.15% of normally distributed data lies to the left of $0{.}5$.

When GPT-2 was released, computing the GELU exactly was expensive, and the released code therefore used an approximation originally presented by [Page (1977)](https://doi.org/10.2307/2346872). We follow suit here, as we want to create a replica of the original model. However, it is worth mentioning that PyTorch now offers an exact implementation of the GELU so fast that using an approximation is unnecessary.

In [3]:
def gelu(x):
    return 0.5 * x * (1 + torch.tanh((2 / torch.pi) ** 0.5 * (x + 0.044715 * x**3)))

#### 🎓 Task 2.02: Mathematical properties of the GELU

Find the minimal output value of the (approximated) GELU and the input value for which it yields that output. Use a service such as [WolframAlpha](https://www.wolframalpha.com/) for the necessary derivations. What are the main differences between the GELU and the ReLU?

From the WolfframAlpha we obtain : min{0.5 x (1 + tanh((2/π)^0.5 (x + 0.044715 x^3)))}≈-0.170041 at x≈-0.752461. Minimum at y = -0.170041 at x≈-0.752461.

Both of these are activation functions that control if a neurons gets activated or not, like a switch that turns on and off the neurons based on thier output. ReLu = max(0,x) if the output of the neuron is negative it does not get activited and stops learning which makes it hard switch meaning the neuron is either activated or not. On The other GeLu is a softer switch and wieghts the input against the probability of it resulting in a dead neuron.




### Feed-forward network

Next, here is the code for the feed-forward network. Note that we follow the released code and use the name **multi-layer perceptron (MLP)** rather than “feed-forward network”.

In [4]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, config.n_embd * 4)
        self.c_proj = nn.Linear(config.n_embd * 4, config.n_embd)

    def forward(self, x):
        batch_size, seq_len, n_embd = x.shape
        #up projection
        x = self.c_fc(x)
        # shape of x: (batch_size, seq_len, n_embd * 4)
        x = gelu(x)
        # down projection
        x = self.c_proj(x)
        # shape of x: (batch_size, seq_len, n_embd)
        return x                                                                                                                                                                                                                                                                                                                

#### 🎓 Task 2.03: Shape annotations

One of the most common errors in deep learning is a mismatch in tensor dimensions. To avoid this, it is good practice to annotate PyTorch code with shapes. For example, suppose you are given the following code:

In [5]:
f = nn.Linear(5, 7)
x = torch.rand(2, 3, 5)
y = f(x)

The annotation of this code with shapes would look as follows:

In [6]:
f = nn.Linear(5, 7)
# not a tensor variable; needs no annotation

x = torch.rand(2, 3, 5)
# shape of x: [2, 3, 5]

y = f(x)
# shape of y: [2, 3, 7]

Annotate the shapes in the `forward()` method of the feed-forward network. Instead of using actual numbers, refer to dimension sizes by symbolic names such as `n_embd`, `batch_size` (number of samples in a batch of input data) and `seq_len` (length of an input sequence). You can introduce additional names and other notation you find useful. Make your annotations as detailed as you need them to explain how the shapes change from one line to the next.

### Causal mask

Our next goal is to implement the core of the GPT architecture: the multi-head attention mechanism.

Recall that the attention mechanism in the Transformer decoder must be restricted to attending only to previously generated tokens. This type of attention is also called **causal attention**. In practice, we implement it through a masking technique that sets the post-softmax attention weights of future tokens to zero. The following utility function implements such a mask:

In [7]:
def make_causal_mask(n):
    return torch.triu(torch.full((n, n), float("-inf")), diagonal=1)

#### 🧩 Task 2.04: Causal mask

Have a close look at the following code and run it to see the result. What are the shapes of `x` and `mask`? Given that the shapes are different, why does the addition operation in the last line not raise an error? What is the shape of the result?

How does the addition operation implement masking? (Recall that the attention scores are normalised using the softmax function.)

In [8]:
x = torch.rand(1, 2, 3, 3)
mask = make_causal_mask(5)
x + mask[:3, :3]

tensor([[[[0.3023,   -inf,   -inf],
          [0.9442, 0.9310,   -inf],
          [0.8594, 0.9158, 0.1663]],

         [[0.8515,   -inf,   -inf],
          [0.2428, 0.3153,   -inf],
          [0.0812, 0.5921, 0.2571]]]])

1. Shape of x: [1,2,3,3] tensor, like two 3x3 matrix stacked side by side.

2. Shape of mask: (5,5) a 5x5 matrix that has -inf upper diagonal and zeros in diagonal and all the elements below the diagonal.

3. The addition does not raise an error because we slice the mask to 3x3 matrix that is added to the 3x3 matrices in x.
The shape of x + mask[:3,:3] is: (1,2,3,3) == x.shape.

4. when the mask is added to x this converts all the upper diagonal of x into -inf and leaves the lower diagonal untouched. when x + mask is passed into softmax only the first column result in something meaningfull and all other columns turn into zero which is the behavior that we want since the first coulumn represent the first token and all other columns passed through softmax result in a zero the model cannot se future tokens.Rows are tokens and columns are tokens being looked at. 


### Attention mechanism

Here is the code for the multi-head attention mechanism:

In [9]:
class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.n_head = config.n_head
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.register_buffer("mask", make_causal_mask(config.n_ctx), persistent=False)

    def forward(self, x):
        batch_size, seq_len, n_embd = x.shape
        # a scalar variable for the dimension of each head
        head_embd = n_embd // self.n_head
        # shape of x: (batch_size, seq_len, 3*n_embd) after c_attn(x)
        # q, k, v: (batch_size, seq_len, n_embd)
        # .chunk(3, dim=-1) splits the last dimension  3 parts, each of size 3*n_embd/3 = n_embd
        q, k, v = self.c_attn(x).chunk(3, dim=-1)
        # reshape q, k, v to (batch_size, seq_len, n_head, head_embd)
        # n_head = number of attention heads; 
        # head_embd = dimension of each head
        q = q.view(batch_size, seq_len, self.n_head, head_embd)
        k = k.view(batch_size, seq_len, self.n_head, head_embd)
        v = v.view(batch_size, seq_len, self.n_head, head_embd)
        # change the order of dimensions to (batch_size, n_head, seq_len, head_embd) for q, k, v
        q = q.transpose(-2, -3)
        k = k.transpose(-2, -3)
        v = v.transpose(-2, -3)
        # attention scores: (batch_size, n_head, seq_len, seq_len)
        x = q @ k.transpose(-1, -2)
        # scale the attention scores by the square root of the head dimension
        # boradcating will scale each attention score by the same factor, which is the square root of the head dimension
        x = x / head_embd**0.5
        # add the causal mask to the attention scores
        # broadcasting will automatically expand the mask to match the shape of x
        x = x + self.mask[:seq_len, :seq_len]
        # apply softmax to get the attention weights
        x = torch.softmax(x, dim=-1)
        # compute the attention output by multiplying the attention weights with v
        # broadcasting will automatically expand the attention weights to match the shape of v
        x = x @ v
        # reshape x back to (batch_size, seq_len, n_embd)
        x = x.transpose(-2, -3).contiguous()
        #  contiguous() continues memeory block after transpose 
        x = x.view(batch_size, seq_len, n_embd)
        # down projection
        x = self.c_proj(x)
        return x

#### 🎓 Task 2.05: Multi-head attention

Trace the input `x` through the `forward()` method line by line and annotate the shapes of all tensor variables. Identify all lines that rely on broadcasting.

### Layer normalisation

As mentioned above, the inputs to both the feed-forward network and the multi-head attention mechanism undergo **layer normalisation**. This normalises the inputs to have zero mean and unit variance across the activations. [Ba et al. (2016)](https://doi.org/10.48550/arXiv.1607.06450) introduce two trainable parameters (called $\gamma$ and $\beta$ in the paper) that allow the network to learn an appropriate scale and shift for the normalised values.

We implement layer normalisation as follows:

In [10]:
class LayerNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.g = nn.Parameter(torch.ones(config.n_embd))
        self.b = nn.Parameter(torch.zeros(config.n_embd))

    def forward(self, x):
        # compute the mean and variance of x along the last dimension, keeping the dimensions for broadcasting
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(unbiased=False, dim=-1, keepdim=True)
        return self.g * (x - mean) / torch.sqrt(variance + 1e-05) + self.b

#### 🧩 Task 2.06: Layer normalisation

What is the relevance of the `keepdim=True` keyword argument in the `mean()` and `var()` functions? What would happen if we omitted it?

What is the relevance of the constant 1e-05? What could happen if we omitted it?

1. we are calculating mean and variance of a along the last dim and keepdim = true preserves the strucute for broadcasting, without it we would a scalar instead of vector.

2. The variance can become extremly small and by adding the constant 1e-15 we are preventing zero division. 

### Decoder block

We now combine the feed-forward network, the multi-head attention mechanism and the layer normalisation into a decoder block.

In [11]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config)
        self.attn = Attention(config)
        self.ln_2 = LayerNorm(config)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

#### 🎓 Task 2.07: Pre-norm and post-norm architectures

The original Transformer ([Vaswani et al., 2017](https://arxiv.org/abs/1706.03762)) is a “post-norm architecture”, where the normalisation is applied **after** each residual block. In contrast, GPT-2 is a “pre-norm architecture”, where the normalisation is applied **before**. Find the passage in Section&nbsp;2.3 of [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) that reports on this modification.

[Xiong et al. (2020)](https://arxiv.org/pdf/2002.04745) compare pre-norm and post-norm architectures empirically. Read the abstract of their paper and summarise their main findings. According to these findings, what are the benefits of the pre-norm architecture?

The passage:

"We use a Transformer (Vaswani et al., 2017) based architecture for our LMs. The model largely follows the details
of the OpenAI GPT model (Radford et al., 2018) with a
Parameters Layers dmodel
117M 12 768
345M 24 1024
762M 36 1280
1542M 48 1600
Table 2. Architecture hyperparameters for the 4 model sizes.
few modifications. Layer normalization (Ba et al., 2016)
was moved to the input of each sub-block, similar to a
pre-activation residual network (He et al., 2016) and an
additional layer normalization was added after the final selfattention block."


The pre-norm architecture removes the need of warm up stage which a way to carefully designing the learning rate and the gradient at the output stage comming out of the FNN is well behaved. In post-LN structure the model need the warmup stage which improves the performance but this adds more parameter to the optimization problem sice in increases the number of hyperparameter and even with warm up stage the gradient at the ouput can large with leads to large learning rate and unstable model. It is intresting to know that a large learning rate can be problem. 

### Model

We now have almost all components in place to complete the implementation of the GPT-2 model. The only thing  missing are the position embeddings. These simply associate an embedding vector with every position in the context window. To set them up, we first define another utility function:

In [12]:
def make_positions(n):
    return torch.arange(n, dtype=torch.long)

We then code the complete model as follows:

In [13]:
class Model(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.n_vocab, config.n_embd)
        self.wpe = nn.Embedding(config.n_ctx, config.n_embd)
        self.h = nn.Sequential(*(Block(config) for _ in range(config.n_layer)))
        self.ln_f = LayerNorm(config)
        self.lm_head = nn.Linear(config.n_embd, config.n_vocab, bias=False)
        self.register_buffer("pos", make_positions(config.n_ctx), persistent=False)

    def forward(self, x):
        batch_size, seq_len = x.shape
        wte = self.wte(x)
        wpe = self.wpe(self.pos[:seq_len])
        x = wte + wpe
        x = self.h(x)
        x = self.ln_f(x)
        x = self.lm_head(x)
        return x

#### 🧩 Task 2.08: Buffers

Our implementation registers the vector of positions as a buffer. (Earlier, we also registered the causal mask as a buffer.) Consult the PyTorch documentation to determine the benefits of registering a tensor as a buffer, in contrast to computing it in the `forward()` method.

Using the buffer allows us to distinguish between the model state and the models paramters. The models state is not updated by the gradient and thus it makes no sense for us to recompute many time inside of the forward method. On the other hand something like the mask does not change over time nor it is updated by the gradient so we save it into the buffer.

#### 🎓 Task 2.09: Number of trainable parameters

The model we have implemented is the smallest one presented by [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf). But how many trainable parameters exactly does it have? Interestingly, the number originally reported by the authors is wrong! What number did they report?

Your task is to write code to compute the number of parameters yourself. This should only take 1–3 lines of code. What number do you get when you apply this code to a fresh model instance?

[Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) followed the original Transformers paper ([Vaswani et al., 2017](https://doi.org/10.48550/arXiv.1706.03762)) and shared the trainable weights between the token embedding and the final linear layer. Implement this weight sharing strategy. (Hint: This only requires one line of code.) Then, re-compute the number of trainable parameters for the modified model. What number do you get now? How large is the reduction caused by the weight sharing?

In [14]:
model = Model(Config())
parameters = sum(p.numel() for p in model.parameters())
parameter_reported = 117_783_552
print(f"Number of parameters: {parameters:,}")
print("procent error: ", (parameters - parameter_reported) / parameter_reported * 100  )

Number of parameters: 163,037,184
procent error:  38.421011449883935


In [15]:
model = Model(Config())
model.lm_head.weight = model.wte.weight
parameters_shared = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {parameters_shared:,}")
print("reduction by sharing the weights of the input and output embeddings: ", (1 - parameters_shared / parameters) * 100, "%")

Number of parameters: 124,439,808
reduction by sharing the weights of the input and output embeddings:  23.673971208923728 %


#### Answer
According to [Radford et al. (2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) the model had 117m parameter but the calulation above roughly 163m parameters approximately 38% wrong.

By sharing weights between the token embedding and final linear layer we the total number of paramters become 124m, a reduction of 23% which close to the reported number by the original paper.

## Part 2: Load pre-trained weights

Now that you have a complete implementation of the GPT-2 model in place, you can instantiate it by loading the pre-trained weights released by OpenAI. These weights were originally provided in the TensorFlow format. For this lab, we have re-packaged them as a single file in NumPy’s `.npz` archive format. We can load it as follows:

In [16]:
import numpy as np

pretrained = np.load("gpt-2-pretrained.npz",allow_pickle=True)
#for name, param in model.named_parameters():
#    print(name, param.shape)

The result `pretrained` is a dictionary mapping names to NumPy arrays. When you print the names, you will see that they correspond to the attributes of our network modules, even though the names differ. For example, the array `h0.attn.c_attn.b` holds the biases (`b`) of the `c_attn` linear layer in the attention mechanism (`attn`) of the first transformer block (`h0`).

#### 🎓 Task 2.10: Load pre-trained weights

Create a model from the pre-trained weights. To do this, you need to instantiate a fresh model and write the contents of each array from the `npz` archive with the pre-trained weights into the corresponding tensor. To make this a bit easier, here is a utility function that copies data from a NumPy array `source` to a PyTorch tensor `target`:

In [17]:
def copy_weights(source: np.ndarray, target: torch.Tensor):
    assert source.shape == target.shape
    with torch.no_grad():
        target.copy_(torch.tensor(source, dtype=torch.float32))

You can start from this skeleton code:

In [18]:
def from_pretrained() -> Model:
    # map h.0 --> h0
    # map .w --> .weight
    # ingnore the special parameters wpe, wte, ln_f.g, ln_f.b
    model = Model(Config())
    model_paramters = dict(model.named_parameters())
    pretrained = np.load("gpt-2-pretrained.npz")

    for name,param in pretrained.items(): 
        if name in ("wpe", "wte"):
            mapped = name + ".weight"
        elif name.startswith("ln_f"):
            mapped = name

        elif name[0] == "h":
            block = name[1:name.index(".")]
            rest = name[name.index(".") + 1:]
            if ".ln_" in name:
                mapped = f"h.{block}.{rest}"
            
            elif rest.endswith(".w"):
                param = param.T
                mapped = f"h.{block}.{rest[:-2]}.weight"
            
            elif rest.endswith(".b"):
                mapped = f"h.{block}.{rest[:-2]}.bias"
        else:
            print(f"Parameter {name} not mapped.")
            continue
        if mapped in model_paramters:
            copy_weights(param, model_paramters[mapped])
        else:
            print(f"Parameter {mapped} not found in the model.")
    model.lm_head.weight = model.wte.weight
    return model

**Important:** One technical detail to note is that PyTorch stores the weights of linear layers in a transposed form. For example, a linear layer created as `nn.Linear(2, 3)` has a weight matrix of shape [3, 2].

## Part 3: Put the model to use

In the third and final part of this lab, you will use the pre-trained model to generate text and evaluate it on a standard benchmark.

### Sampling-based text generation

The easiest way to generate text with a language model is by using a **greedy approach**. This method works by creating text one token at a time. At each step, the model takes the previously generated text (called the **context**) as input and adds the token with the highest output logit as a new token. The code in the next cell defines a function `generate()` that forms the core of a greedy generator:

In [19]:
def generate(model, context, context_size=1024, n_tokens=20,temperature=1.0,top_k=10):
    # generate n_tokens tokens autoregressively, starting from the given context
    # predict the next token, append it to the context, and repeat
    for _ in range(n_tokens):
        # context ist the input to the model
        context = context[:, -context_size:]
        with torch.no_grad():
            # apply the model to get the logits
            logits = model(context)[:, -1, :]
        # scale the logits by T to control randomness
        logits = logits / temperature
        # if top k is specified 
        if top_k is not None:
            # chose the top k logit and set the rest to -inf
            top_values, _ = torch.topk(logits, top_k)
            logits[logits < top_values[:, -1, None]] = float("-inf")
        # sample the next token from the distribution defined by the logits
        next_idx = torch.multinomial(logits.softmax(dim=-1), num_samples=1)
        # append the next token to the context for the next iteration
        context = torch.cat([context, next_idx], dim=-1)
    return context

To use this function with an actual text input, you need a tokeniser to first encode the text into a vector of token IDs, and later decode the generated `context` into new text. The reference implementation of the GPT-2 tokeniser is in the library `tiktoken`. The code in the next cell sets up the tokeniser, loads the pretrained model from Task&nbsp;2.10, and then defines a helper function that handles the encoding and decoding.

In [24]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
model = from_pretrained()

def generate_helper(text, context_size=1024, n_tokens=20):
    context = torch.tensor([tokenizer.encode(text)], dtype=torch.long)
    context = generate(model, context, context_size=context_size, n_tokens=n_tokens)
    return tokenizer.decode(context[0].tolist())

You can use this helper function to generate text as follows:

In [21]:
generate_helper("tell me a joke")

"tell me a joke about my mom when we were growing up, I don't know, but we had a little bit of a bad feeling about it. She didn't like that. She didn't like it. I"

**Tip:** If you did not manage to complete Task&nbsp;2.10, you can still work on this task by using a pretrained GPT-2 model from [Hugging Face](https://huggingface.co/openai-community/gpt2). The next code cell shows how you would instantiate this model. Note that you may have to first install the `transformers` library.

In [22]:
# from transformers import GPT2LMHeadModel
# model = GPT2LMHeadModel.from_pretrained("gpt2")
# logits = model(context).logits[:, -1, :]

#### 🎓 Task 2.11: Sampling-based text generation

 The greedy approach to text generation is not very interesting for practical applications because it always chooses the most likely token, leading to predictable and less creative results. Your task is to modify the code for the `generate()` function to use a **sampling-based approach** instead. In this approach, the next token is chosen randomly based on the probabilities assigned by the model (softmax-normalised logits), treating them as a categorical distribution over the token vocabulary. Additionally, your code should include two common techniques to improve sampling:
 
 * **temperature scaling**, which lets the user control the randomness of the sampling
 * **top-$k$ sampling**, which limits the sampling to the top-$k$ most likely tokens, ignoring less probable ones

### Notes

* **temperature scaling**, scale down the logits with T and sample from the distribution of the logits .
* **top-$k$ sampling**, only keep the top-k logits and set the rest to -inf.

### Evaluating the pretrained model

If you have experimented with your pretrained GPT-2 model, you will have noticed that its ability to generate useful text is somewhat limited. By today’s standards, GPT-2 is a small model with modest capabilities. However, it can still be helpful for certain tasks, such as text autocompletion, generating filler text, or answering simple questions. To rigourosly evaluate language models, researchers often use standard benchmark datasets. Creating these benchmarks is a discipline of its own, and they tend to become increasingly challenging as models continue to improve.

In the final task of this lab, you will evaluate GPT-2’s performance on a small subset of the [HellaSwag dataset](https://rowanzellers.com/hellaswag/), which was published in the same year as GPT-2 itself (2019). HellaSwag is designed to test a model’s ability to perform commonsense reasoning in challenging contexts. Unlike simpler benchmarks, HellaSwag presents scenarios where the correct text completion depends on semantic relationships between events and on world knowledge. This makes it a good choice for assessing the ability of language models to go beyond surface-level patterns and produce meaningful, context-aware predictions.

#### 🎓 Task 2.12: Evaluating the pretrained model

Read the [HellaSwag website](https://rowanzellers.com/hellaswag/) to get some background on the benchmark. How does a sample from the dataset look like? What is an expected prediction? How does the benchmark allow us to score models? What is the random baseline? What is the human performance reported on the task?

The next cell contains code for evaluating your pretrained model on a small sample from HellaSwag. You will also need a tokenizer. The HellaSwag subset is in the file `hellaswag-mini.jsonl`. Inspect that file to understand the format. Next, read the code and explain how it works. Specifically, how does the code compute the score of individual endings? In the call to `cross_entropy()`, why are the tensors sliced in this specific way?

Finally, what overall score does the pretrained GPT-2 model get on this benchmark? How does that score compare to the random baseline and the human performance?

### Answer

1. The benchmark aims to evaluate a model ability to on commonsense inference, previous to this most of the benchmark had only a statistical nature but Hellswag mixes this statistical nature with how the physical world works to test if the model really understands the world or is it merely really good at extracting statistical patterns.

    A sample from this benchamrk is a discription of a mundane scenario framed a multiple choice question where 3 out 4 the four alternatives are machine generated text and one is the correct answer. The inclusion of the  machine generated alternatives aim to penalise an answer that is only based on statistical pattern recognition.

    Ex "A man is sitting at a piano. He opens the lid, adjusts his bench, and..."
    Correct answer : Correct Ending: "...begins to play a classical piece."
    distractor: ."..starts to eat the piano strings with a fork."


2. A expected prediction is the most logically sound completion of sentence describing a real world scenario.


3. The bechmark works like a standardized exam that every model takes. Every model get a question and four option and the answers the models provid are compared to the ground truth verfied by humans.

4. The random baseline is the score the model would get if it had no intelligence and rellied on random gusses.
5. The human performance reported in the paper is above 95%.


The score calculation mechanism tests the reaction of the model to an input instead of making the model generate text. The cross entropy measures the distance between two probability distributions. Low score means the model expected the output and is not supprised and in the opposite scenario if the score is high the model is supprised by the tokens. The code calculates this scores for all four option in ending score and uses min(ending_score) to pick one with the lowest lost.

The logit at position t is the prediction at position t+1.

context[0, -len(suffix) :] this the target context, the tokens of the suffix if the ending has 5 tokens the combined model, these are the correct answers we are testing the model against.

logits[0, -len(suffix) - 1 : -1] the model prediction that correspond to the target tokens. -len(suffix) - 1 : -1 shifts the logits one position to the left so they align with ground truth.


The model scored 30,47% which mush lower than human level accuracy > 95% and better than random baseline for gpt2 which is 25%.

In [25]:
import json

with open("hellaswag-mini.jsonl") as f:
    n_correct = 0
    n_total = 0
    for line in f:
        sample = json.loads(line)
        prefix = tokenizer.encode(sample["ctx"])
        ending_scores = []
        for i, ending in enumerate(sample["endings"]):
            suffix = tokenizer.encode(" " + ending)
            context = torch.tensor([prefix + suffix], dtype=torch.long)
            with torch.no_grad():
                logits = model(context)
                ending_score = torch.nn.functional.cross_entropy(
                    logits[0, -len(suffix) - 1 : -1], context[0, -len(suffix) :]
                )
            ending_scores.append((ending_score, i))
        predicted = min(ending_scores)[1]
        n_correct += int(predicted == sample["label"])
        n_total += 1
    print(f"Accuracy: {n_correct / n_total:.2%}")

Accuracy: 30.47%


**🥳 Congratulations on finishing lab&nbsp;2!**